# Long-Term Memory Agent
## Remember Across Threads — InMemoryStore for Persistent Facts
⏱ ~12 min

**Long-term memory** is the pattern that separates a stateless chatbot from an agent that actually knows you. Standard LangGraph state resets on every `invoke` — your name, preferences, and past conversations vanish. Long-term memory breaks that boundary: facts extracted from one conversation thread are persisted to a store and retrieved in every future thread, so the agent grows smarter over time.

By the end of this workshop you will understand *why* memory matters, *how* the retrieve-respond-store loop works, and *how* to build a LangGraph agent that remembers users across sessions.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why long-term memory, and what kinds exist? |
| 2 | **InMemoryStore** — The building block: put, get, search |
| 3 | **Namespaces** — Isolating memories per user |
| 4 | **The Memory Graph** — retrieve → respond → store loop |
| 5 | **Cross-Thread Recall** — Thread-1 stores; Thread-2 retrieves |
| 6 | **Multiple Users** — Namespace isolation in practice |
| 7 | **Memory Quality** — What gets stored and why |
| ★ | **Extensions** — Decay, scoring, persistent backends (bonus) |

---

### Prerequisites
- Python 3.10+ (a `.venv` with `requirements.txt` already installed)
- An `OPENAI_API_KEY` in `.env` (or Colab Secrets)

### Key References
> Park, J. S., O'Brien, J., Cai, C. J., et al. (2023). *Generative Agents: Interactive Simulacra of Human Behavior.* UIST 2023. https://arxiv.org/abs/2304.03442
>
> Zhong, W., Guo, L., Gao, Q., et al. (2024). *MemoryBank: Enhancing Large Language Models with Long-Term Memory.* AAAI 2024. https://arxiv.org/abs/2305.10250
>
> LangGraph Memory Store docs: https://langchain-ai.github.io/langgraph/concepts/memory/
>
> LangGraph InMemoryStore API: https://langchain-ai.github.io/langgraph/reference/store/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import json
import os
from typing import TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid. "
    "Local: add it to .env. Colab: add it in the Secrets panel."
)
print(f"OPENAI_API_KEY present: {bool(key)} (starts with sk-: {key.startswith('sk-')})")

---

## Part 1 — Concepts: Why Long-Term Memory?

---

### The Statefulness Gap

Every LangGraph `invoke` call starts with a clean slate. Thread state lives only for the duration of a single run — when the process ends (or even between `invoke` calls), it is gone. This is fine for one-shot tasks but breaks down for agents that are supposed to know users over time.

Long-term memory plugs this gap by persisting facts *outside* the graph state, in a store that survives across threads and restarts.

---

### Three Types of Agent Memory

| Type | What it stores | Lifetime | LangGraph primitive |
|------|---------------|----------|---------------------|
| **Short-term (in-context)** | Current conversation messages | Single `invoke` | `state["messages"]` |
| **Cross-thread (long-term)** | Facts about users, preferences, history | Across threads | `InMemoryStore` / `SqliteStore` |
| **Persistent (durable)** | Same as above but survives restarts | Indefinite | `SqliteStore`, Postgres, Redis |

---

### Cognitive Memory Taxonomy

Memory research in cognitive science distinguishes three kinds. Agent memory systems often mirror this taxonomy:

| Memory kind | Definition | Agent analogy |
|-------------|-----------|---------------|
| **Episodic** | Specific past events ("the user said X on Tuesday") | Stored conversation excerpts |
| **Semantic** | General world knowledge ("Alice is a Python developer") | Extracted facts in the store |
| **Procedural** | How to do things ("always respond concisely") | System prompt instructions derived from preferences |

This workshop focuses on **semantic memory** — extracting and retrieving factual statements about users.

---

### Memory Storage Backends

| Backend | Persistence | Semantic search | When to use |
|---------|------------|-----------------|-------------|
| `InMemoryStore` | Process lifetime only | Yes (embedding-based) | Development, notebooks, demos |
| `SqliteStore` | File on disk | Yes | Single-server production |
| PostgreSQL + pgvector | Durable, scalable | Yes | Multi-server production |
| Redis | Fast, in-memory | With RediSearch module | Low-latency retrieval |

---

### Memory Architecture — Full Picture

```
CONVERSATION TURN (every invoke)
────────────────────────────────────────────────────────────────

  User message
       │
       ▼
  ┌─────────────────────┐
  │  retrieve_memories  │  store.search(namespace, query, limit=5)
  │                     │  → semantic search over all stored facts
  └──────────┬──────────┘
             │  top-k relevant facts
             ▼
  ┌─────────────────────┐
  │      respond        │  SystemMessage with memories injected
  │                     │  + conversation history
  │                     │  → LLM produces personalized response
  └──────────┬──────────┘
             │  response text
             ▼
  ┌─────────────────────┐
  │   store_memories    │  LLM extracts 1-3 facts from the turn
  │                     │  store.put(namespace, key, {"fact": ...})
  └──────────┬──────────┘
             │
            END


CROSS-THREAD RECALL
────────────────────────────────────────────────────────────────

  Thread-1 (past)             Thread-2 (now)
  ─────────────               ─────────────────
  store.put(ns, k, fact) ───► InMemoryStore ───► store.search(ns, query)
                                                         │
                                                   facts injected
                                                   into context
```

---

## Part 2 — InMemoryStore: The Building Block

---

`InMemoryStore` is LangGraph's cross-thread key-value store. It lives outside graph state and is shared across all threads in a process. Three operations drive everything:

| Operation | Signature | Purpose |
|-----------|-----------|--------|
| `store.put` | `(namespace, key, value: dict)` | Write a fact |
| `store.get` | `(namespace, key)` | Read one fact by key |
| `store.search` | `(namespace, query=str, limit=N)` | Semantic search over facts |

**Namespace** is a tuple like `("memories", "user-alice")` — it scopes facts per user. Alice's memories never appear in Bob's search results.

**Semantic search** means `store.search` uses embeddings under the hood — it finds relevant facts even when the query wording differs from the stored text.

In [ ]:
# ─── 2-A: Basic put and get ───────────────────────────────────────────────────
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
ns = ("memories", "demo-user")

# Write three facts
store.put(ns, "fact-0", {"fact": "User is a Python developer"})
store.put(ns, "fact-1", {"fact": "User prefers concise responses"})
store.put(ns, "fact-2", {"fact": "User is building a LangGraph agent"})

# Read one back by key
item = store.get(ns, "fact-0")
print(f"Direct get by key: {item.value}")
print()

In [ ]:
# ─── 2-B: Semantic search ─────────────────────────────────────────────────────
# store.search uses embeddings — finds relevant facts even when keywords differ.

queries = [
    "what programming language does the user prefer?",
    "how should I communicate with the user?",
    "what is the user working on?",
]

for query in queries:
    results = store.search(ns, query=query, limit=2)
    top = results[0].value["fact"] if results else "(no results)"
    print(f"Query: '{query}'")
    print(f"  Top match: {top}")
    print()

In [ ]:
# ─── 2-C: Overwriting a fact and listing all items ────────────────────────────
# put() is idempotent: the same key overwrites the previous value.

# Update an existing key
store.put(ns, "fact-1", {"fact": "User prefers bullet-point responses, not prose"})

# List all facts in the namespace
all_items = store.search(ns, query="everything about the user", limit=20)
print(f"All stored facts ({len(all_items)} items):")
for item in all_items:
    print(f"  [{item.key}]  {item.value['fact']}")

### Exercise 1 — Explore the Store

Add five facts about a fictional user named Bob (interests, job, communication style, location, a quirk). Then run three queries and verify that semantic search retrieves the right fact for each query.

**Goal:** Observe that even when your query wording does not match the stored fact verbatim, the correct fact still comes back. This is the power of embedding-based search.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
ex1_store = InMemoryStore()
bob_ns = ("memories", "user-bob")

# TODO: Add 5 facts about Bob
# ex1_store.put(bob_ns, "fact-0", {"fact": "..."})

# TODO: Run 3 queries and print the top result for each
ex1_queries = [
    "TODO: first query",
    "TODO: second query",
    "TODO: third query",
]

for q in ex1_queries:
    if q.startswith("TODO"):
        continue
    results = ex1_store.search(bob_ns, query=q, limit=1)
    print(f"Q: {q}")
    print(f"A: {results[0].value['fact'] if results else '(no results)'}")
    print()

---

## Part 3 — Namespaces: Isolating Memories Per User

---

A **namespace** is a tuple of strings that acts like a folder path for stored facts. The convention used in this example is:

```python
NAMESPACE = ("memories", user_id)
# e.g. ("memories", "user-alice")  or  ("memories", "user-bob")
```

Facts stored under one namespace are invisible when searching another — this is how you guarantee that Alice's preferences never appear in Bob's chat.

### Namespace Design Patterns

| Pattern | Namespace | Use case |
|---------|-----------|----------|
| Per-user | `("memories", user_id)` | Personal assistant, CRM |
| Per-user per-topic | `("memories", user_id, "preferences")` | Complex domains |
| Per-session | `("sessions", session_id)` | Ephemeral context |
| Global shared | `("knowledge", "company")` | Shared knowledge base |

In [ ]:
# ─── 3-A: Namespace isolation demo ───────────────────────────────────────────
ns_store = InMemoryStore()

alice_ns = ("memories", "user-alice")
bob_ns = ("memories", "user-bob")

ns_store.put(alice_ns, "fact-0", {"fact": "Alice is a Python developer"})
ns_store.put(alice_ns, "fact-1", {"fact": "Alice lives in San Francisco"})
ns_store.put(bob_ns, "fact-0", {"fact": "Bob is a data scientist"})
ns_store.put(bob_ns, "fact-1", {"fact": "Bob prefers R over Python"})

# Query Alice's namespace
alice_results = ns_store.search(alice_ns, query="what does the user do for work?", limit=3)
print("Alice's namespace results:")
for r in alice_results:
    print(f"  {r.value['fact']}")

print()

# Query Bob's namespace — Alice's facts never appear here
bob_results = ns_store.search(bob_ns, query="what does the user do for work?", limit=3)
print("Bob's namespace results:")
for r in bob_results:
    print(f"  {r.value['fact']}")

print()
print("Isolation confirmed: Alice's facts do not appear in Bob's results.")

In [ ]:
# ─── 3-B: Multi-level namespaces ──────────────────────────────────────────────
# When a user has many facts, sub-namespacing by topic keeps retrieval precise.

ml_store = InMemoryStore()
USER = "user-carol"

# Store facts in two sub-namespaces
prefs_ns = ("memories", USER, "preferences")
work_ns = ("memories", USER, "work")

ml_store.put(prefs_ns, "p0", {"fact": "Carol prefers dark mode"})
ml_store.put(prefs_ns, "p1", {"fact": "Carol responds best to bullet points"})
ml_store.put(work_ns, "w0", {"fact": "Carol is a machine learning engineer"})
ml_store.put(work_ns, "w1", {"fact": "Carol's team uses PyTorch for model training"})

# Search only within the 'work' namespace
results = ml_store.search(work_ns, query="what frameworks does the user use?", limit=3)
print("Work namespace only:")
for r in results:
    print(f"  {r.value['fact']}")

---

## Part 4 — The Memory Graph: Retrieve → Respond → Store

---

The three-node graph is the core pattern of this example. Each node has a single responsibility:

| Node | Input from state | What it does | Output to state |
|------|-----------------|--------------|----------------|
| `retrieve_memories` | `messages[-1]` (latest user message) | Semantic search over the store; returns top-k facts | `memories: list[str]` |
| `respond` | `memories`, `messages` | Injects memories into system prompt; calls LLM | `response: str` |
| `store_memories` | `messages`, `response` | Asks LLM to extract facts; calls `store.put` for each | (side effect only) |

### State Schema

```python
class MemoryState(TypedDict):
    thread_id: str        # unique per conversation turn
    messages:  list[str]  # conversation history for this turn
    memories:  list[str]  # facts retrieved from the store
    response:  str        # LLM's answer
```

### Graph Topology

```
START
  │
  ▼
retrieve_memories   ← store.search(namespace, query=messages[-1], limit=5)
  │
  ▼
respond             ← SystemMessage(memories injected) + HumanMessage(history)
  │                   → LLM produces personalized response
  ▼
store_memories      ← LLM extracts facts → store.put(namespace, key, {"fact": ...})
  │
  ▼
END
```

In [ ]:
# ─── 4-A: Define state, store, and LLM ───────────────────────────────────────
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langgraph.store.memory import InMemoryStore

USER_ID = "user-alice"
NAMESPACE = ("memories", USER_ID)

graph_store = InMemoryStore()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

SYSTEM_PROMPT = (
    "You are a helpful assistant with long-term memory about the user.\n"
    "Retrieved memories:\n{memories}\n\n"
    "Use these memories to personalize your response. "
    "If no memories are retrieved yet, respond helpfully and ask for context."
)


class MemoryState(TypedDict):
    thread_id: str
    messages: list[str]
    memories: list[str]
    response: str


print("State, store, and LLM ready.")

In [ ]:
# ─── 4-B: Define the three graph nodes ───────────────────────────────────────


def retrieve_memories(state: MemoryState) -> dict:
    """Search the store for facts relevant to the latest user message."""
    query = state["messages"][-1] if state["messages"] else ""
    results = graph_store.search(NAMESPACE, query=query, limit=5)
    memories = [r.value["fact"] for r in results]
    print(f"  [retrieve] found {len(memories)} memory/memories")
    return {"memories": memories}


def respond(state: MemoryState) -> dict:
    """Generate a response with retrieved memories injected into the system prompt."""
    memory_text = "\n".join(f"- {m}" for m in state["memories"]) or "None yet"
    system = SYSTEM_PROMPT.format(memories=memory_text)
    history = [HumanMessage(content=msg) for msg in state["messages"]]
    resp = llm.invoke([SystemMessage(content=system)] + history)
    return {"response": resp.content}


def store_memories(state: MemoryState) -> dict:
    """Extract facts from the conversation and persist them to the store."""
    extract_prompt = (
        f"Extract 1-3 factual statements about the user from this conversation.\n"
        f"Messages: {state['messages']}\nAssistant response: {state['response']}\n"
        f'Return ONLY a JSON array of strings, e.g. ["User is a Python developer"]'
    )
    result = llm.invoke([HumanMessage(content=extract_prompt)])
    try:
        facts = json.loads(result.content)
        for i, fact in enumerate(facts[:3]):
            key = f"{state['thread_id']}-fact-{i}"
            graph_store.put(NAMESPACE, key, {"fact": fact})
            print(f"  [store] saved: {fact}")
    except (json.JSONDecodeError, TypeError) as e:
        print(f"  [store] fact extraction failed: {e}")
    return {}


print("Node functions defined.")

In [ ]:
# ─── 4-C: Compile the graph ───────────────────────────────────────────────────

graph = StateGraph(MemoryState)
graph.add_node("retrieve", retrieve_memories)
graph.add_node("respond", respond)
graph.add_node("store", store_memories)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "respond")
graph.add_edge("respond", "store")
graph.add_edge("store", END)

app = graph.compile()
print("Graph compiled successfully.")
print("Nodes:", ["retrieve", "respond", "store", "END"])
print("Edges: retrieve → respond → store → END")

---

## Part 5 — Cross-Thread Recall: Memory in Action

---

This is the payoff. We run two separate `invoke` calls — two different conversation threads. Thread-1 introduces the user. Thread-2 asks questions that can only be answered if the memories from Thread-1 were saved and retrieved.

**Without** long-term memory: Thread-2 gets a generic answer.
**With** long-term memory: Thread-2 gets a personalized, context-aware answer.

In [ ]:
# ─── 5-A: Thread-1 — introduce the user ──────────────────────────────────────
print("=" * 60)
print("THREAD-1: Introduction")
print("=" * 60)

result_t1 = app.invoke(
    {
        "thread_id": "thread-1",
        "messages": [
            "Hi! I'm Alice. I'm a Python developer working on LangGraph.",
            "I love building agentic systems and I prefer concise responses.",
        ],
        "memories": [],
        "response": "",
    }
)

print(f"\nMemories retrieved at start of Thread-1: {result_t1['memories']}")
print(f"Response: {result_t1['response'][:300]}")

In [ ]:
# ─── 5-B: Thread-2 — recall and personalize ───────────────────────────────────
# A completely separate thread. The agent has no messages from Thread-1
# in its state — but it has the facts stored in the InMemoryStore.

print("=" * 60)
print("THREAD-2: Recall and personalization")
print("=" * 60)

result_t2 = app.invoke(
    {
        "thread_id": "thread-2",
        "messages": [
            "What do you know about me?",
            "Can you recommend a next step for my LangGraph project?",
        ],
        "memories": [],
        "response": "",
    }
)

print(f"\nMemories retrieved at start of Thread-2: {result_t2['memories']}")
print(f"\nResponse: {result_t2['response'][:500]}")

In [ ]:
# ─── 5-C: Inspect all stored facts after two threads ─────────────────────────
all_facts = graph_store.search(NAMESPACE, query="everything", limit=20)
print(f"Total facts in store for {USER_ID}: {len(all_facts)}")
print()
for item in all_facts:
    print(f"  [{item.key}]  {item.value['fact']}")

### Exercise 2 — Add a Third Thread

Add a Thread-3 that continues the conversation with a completely new topic (e.g., asking for advice on a personal project, or asking about a hobby). Verify that:
1. The memories from Threads 1 and 2 are retrieved.
2. The response reflects those memories.
3. New facts are extracted and stored.

**Challenge:** After Thread-3, inspect the store and count how many unique facts are now stored.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

# TODO: Invoke the app with a thread-3 that uses new messages.
# result_t3 = app.invoke({
#     "thread_id": "thread-3",
#     "messages": ["TODO: your messages here"],
#     "memories": [],
#     "response": "",
# })

# TODO: Print result_t3["memories"] — were facts from earlier threads retrieved?
# TODO: Print result_t3["response"]

# TODO: Inspect the store again and count total facts
# all_facts = graph_store.search(NAMESPACE, query="everything", limit=50)
# print(f"Total facts after 3 threads: {len(all_facts)}")

---

## Part 6 — Multiple Users: Namespace Isolation in Practice

---

A real system serves many users. The namespace pattern guarantees isolation: querying Bob's namespace never returns Alice's facts, no matter how semantically similar they are.

This section demonstrates how to parameterize the graph to work for any user ID.

In [ ]:
# ─── 6-A: Multi-user graph factory ───────────────────────────────────────────
# Build a fresh graph wired to a shared store but parameterized by user_id.

multi_store = InMemoryStore()


def build_user_app(user_id: str, shared_store: InMemoryStore):
    """Return a compiled graph that scopes all memory operations to user_id."""
    namespace = ("memories", user_id)

    def _retrieve(state: MemoryState) -> dict:
        query = state["messages"][-1] if state["messages"] else ""
        results = shared_store.search(namespace, query=query, limit=5)
        return {"memories": [r.value["fact"] for r in results]}

    def _respond(state: MemoryState) -> dict:
        mem_text = "\n".join(f"- {m}" for m in state["memories"]) or "None yet"
        system = SYSTEM_PROMPT.format(memories=mem_text)
        history = [HumanMessage(content=msg) for msg in state["messages"]]
        resp = llm.invoke([SystemMessage(content=system)] + history)
        return {"response": resp.content}

    def _store(state: MemoryState) -> dict:
        prompt = (
            f"Extract 1-2 facts about the user. Messages: {state['messages']}\n"
            f"Response: {state['response']}. Return a JSON array of strings."
        )
        result = llm.invoke([HumanMessage(content=prompt)])
        try:
            facts = json.loads(result.content)
            for i, fact in enumerate(facts[:2]):
                shared_store.put(namespace, f"{state['thread_id']}-fact-{i}", {"fact": fact})
        except (json.JSONDecodeError, TypeError):
            pass
        return {}

    g = StateGraph(MemoryState)
    g.add_node("retrieve", _retrieve)
    g.add_node("respond", _respond)
    g.add_node("store", _store)
    g.set_entry_point("retrieve")
    g.add_edge("retrieve", "respond")
    g.add_edge("respond", "store")
    g.add_edge("store", END)
    return g.compile()


alice_app = build_user_app("user-alice", multi_store)
bob_app = build_user_app("user-bob", multi_store)
print("Multi-user apps ready.")

In [ ]:
# ─── 6-B: Run both users and verify isolation ─────────────────────────────────
# Alice and Bob each have one introduction thread.
# Then each asks the same question — they should get different personalized answers.

for user_label, user_app, intro_msgs in [
    (
        "ALICE",
        alice_app,
        ["Hi, I'm Alice. I'm a Python developer and avid rock climber."],
    ),
    (
        "BOB",
        bob_app,
        ["Hey, I'm Bob. I'm a chef who loves experimenting with fermentation."],
    ),
]:
    # Introduction thread
    user_app.invoke(
        {
            "thread_id": f"{user_label.lower()}-intro",
            "messages": intro_msgs,
            "memories": [],
            "response": "",
        }
    )

    # Follow-up thread — same question for both users
    result = user_app.invoke(
        {
            "thread_id": f"{user_label.lower()}-follow",
            "messages": ["What should I try this weekend?"],
            "memories": [],
            "response": "",
        }
    )

    print(f"=== {user_label} ===")
    print(f"Memories: {result['memories']}")
    print(f"Response: {result['response'][:300]}")
    print()

### Exercise 3 — Build a Third User

Add a third user (Carol — a data scientist who loves hiking). Run an introduction thread and a follow-up thread. Then run this isolation check:

```python
# Verify isolation: Carol's facts must not appear in Alice's namespace
alice_facts = multi_store.search(("memories", "user-alice"), query="hiking data science", limit=5)
assert all("Carol" not in f.value["fact"] for f in alice_facts), "Isolation breach!"
print("Isolation check passed.")
```

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────

# TODO: Build carol_app using build_user_app("user-carol", multi_store)

# TODO: Run an introduction thread for Carol

# TODO: Run a follow-up thread for Carol

# TODO: Run the isolation check below
# alice_facts = multi_store.search(("memories", "user-alice"), query="hiking data science", limit=5)
# assert all("Carol" not in f.value["fact"] for f in alice_facts), "Isolation breach!"
# print("Isolation check passed.")

---

## Part 7 — Memory Quality: What Gets Stored and Why

---

The quality of the memory system depends entirely on what the extraction step stores. Poorly extracted facts lead to noise in retrieval; well-extracted facts lead to a growing, accurate user model.

### Common Extraction Failure Modes

| Failure | Symptom | Fix |
|---------|---------|-----|
| **Over-extraction** | Trivial or redundant facts stored | Prompt: "store only durable preferences and important attributes" |
| **Under-extraction** | Key facts missed | Prompt: "always extract name, profession, and preferences" |
| **Hallucinated facts** | LLM invents attributes | Prompt: "only extract facts explicitly stated, not inferred" |
| **Stale facts** | Old preferences override new ones | Key deduplication + timestamp filtering |
| **Noisy retrieval** | Irrelevant facts returned | Smaller `limit`, better query formulation |

### Extraction Prompt Quality Matters

Below we compare a weak and a strong extraction prompt on the same conversation.

In [ ]:
# ─── 7-A: Compare weak vs strong extraction prompts ──────────────────────────

sample_conversation = [
    "Hi! I'm Diana. I work as a UX designer at a startup in Berlin.",
    "I've been learning to code in Python on weekends — mostly for data viz.",
    "I hate jargon and prefer examples over theory.",
]

weak_prompt = f"Extract facts from this text: {sample_conversation}. Return a JSON array."

strong_prompt = (
    f"Extract 2-4 durable facts about the user from this conversation.\n"
    f"Rules:\n"
    f"- Only include facts explicitly stated, never inferred.\n"
    f"- Prefer preferences, skills, background, and goals.\n"
    f"- Avoid transient details (e.g., 'said hello').\n"
    f"Conversation: {sample_conversation}\n"
    f"Return ONLY a JSON array of concise strings."
)

for label, prompt in [("WEAK", weak_prompt), ("STRONG", strong_prompt)]:
    result = llm.invoke([HumanMessage(content=prompt)])
    try:
        facts = json.loads(result.content)
    except json.JSONDecodeError:
        facts = [result.content]
    print(f"--- {label} PROMPT ---")
    for f in facts:
        print(f"  {f}")
    print()

In [ ]:
# ─── 7-B: Key deduplication strategy ─────────────────────────────────────────
# Problem: if you store a new "profession" fact every turn, old ones pile up.
# Solution: derive the key from the fact TYPE, not from thread_id + index.

dedup_store = InMemoryStore()
dedup_ns = ("memories", "user-diana")

# Turn 1 — initial profession fact
dedup_store.put(dedup_ns, "profession", {"fact": "Diana is a UX designer"})
dedup_store.put(dedup_ns, "location", {"fact": "Diana lives in Berlin"})
dedup_store.put(dedup_ns, "learning", {"fact": "Diana is learning Python on weekends"})

# Turn 2 — user says they changed jobs
# Using the same key 'profession' OVERWRITES the old value — no duplicate.
dedup_store.put(dedup_ns, "profession", {"fact": "Diana is now a senior product designer"})

all_diana = dedup_store.search(dedup_ns, query="Diana's job", limit=10)
print(f"Facts stored (dedup): {len(all_diana)}")
for item in all_diana:
    print(f"  [{item.key}] {item.value['fact']}")

print()
print("Note: 'profession' key was overwritten — only the latest value is stored.")

---

## Part 8 ★ — Extensions (Bonus)

---

### Memory Decay

Real memory fades. You can simulate decay by attaching a timestamp to every stored fact and filtering out facts older than a threshold at retrieval time.

```python
import time

store.put(ns, "fact-0", {"fact": "User prefers Python", "stored_at": time.time()})

# At retrieval: filter out stale facts
MAX_AGE_SECONDS = 60 * 60 * 24 * 7  # 1 week
now = time.time()
results = store.search(ns, query=query, limit=10)
fresh = [r for r in results if now - r.value.get("stored_at", now) < MAX_AGE_SECONDS]
```

---

### Persistent Store (SqliteStore)

`InMemoryStore` loses all data when the process ends. For production, swap in `SqliteStore`:

```python
# pip install langgraph-checkpoint-sqlite
from langgraph.store.sqlite import SqliteStore

store = SqliteStore("./memory.db")
# Exact same API: store.put, store.get, store.search
# Data survives restarts — reads it back from memory.db on init.
```

---

### Memory Scoring

Not all memories are equally important. You can extend the stored value with a `weight` field and sort retrieval results by it:

```python
store.put(ns, "core-identity", {
    "fact": "User is a senior ML engineer",
    "weight": 1.0,    # always surface this
    "category": "identity",
})
store.put(ns, "transient-pref", {
    "fact": "User mentioned liking dark chocolate today",
    "weight": 0.2,    # low-value transient fact
    "category": "preference",
})
```

In [ ]:
# ─── 8-A: Memory decay with timestamps ───────────────────────────────────────
import time

decay_store = InMemoryStore()
decay_ns = ("memories", "user-eve")

# Store facts with timestamps
now = time.time()
decay_store.put(decay_ns, "recent", {"fact": "Eve just started a new job", "stored_at": now})
decay_store.put(
    decay_ns,
    "old",
    {
        "fact": "Eve used to work in marketing (years ago)",
        "stored_at": now - 60 * 60 * 24 * 30,  # 30 days ago
    },
)
decay_store.put(
    decay_ns,
    "timeless",
    {
        "fact": "Eve is a Python developer",
        "stored_at": now - 60 * 60 * 24 * 365,  # 1 year ago
    },
)

# Retrieve with decay filtering (keep only facts from last 7 days)
MAX_AGE = 60 * 60 * 24 * 7
results = decay_store.search(decay_ns, query="tell me about Eve", limit=10)
fresh = [r for r in results if now - r.value.get("stored_at", now) < MAX_AGE]
stale = [r for r in results if now - r.value.get("stored_at", now) >= MAX_AGE]

print("Fresh facts (< 7 days):")
for r in fresh:
    print(f"  {r.value['fact']}")

print("\nStale facts (>= 7 days, filtered out):")
for r in stale:
    age_days = (now - r.value["stored_at"]) / (60 * 60 * 24)
    print(f"  {r.value['fact']}  ({age_days:.0f} days old)")

---

## What's Next?

You now understand long-term memory: how to persist facts across threads, isolate users by namespace, and build the retrieve-respond-store loop. Here is where to go deeper:

### Make the memory durable
- Swap `InMemoryStore` for `SqliteStore` so memories survive process restarts. The API is identical — just change the import and constructor.

### Combine with graph checkpointing
- **Example 39 — Checkpoint Resume** (`../39-checkpoint-resume/`): `SqliteSaver` persists the full graph *state* between turns. Combine it with `SqliteStore` for complete statefulness.

### Add human oversight
- **Example 11 — HITL Approval** (`../11-hitl-approval/`): add a human-in-the-loop interrupt before the `store` node so a user can approve or reject extracted facts before they are saved.

### Scale to multiple agents
- **Example 6 — Multi-Agent Graph** (`../6-multi-agent-graph/`): multiple agents can share the same `InMemoryStore` instance, enabling a team of specialists to collectively build and query a shared user model.

### Read the foundational papers
- **Generative Agents** (Park et al., 2023): the architecture that popularized agent memory streams, retrieval scoring, and reflection — https://arxiv.org/abs/2304.03442
- **MemoryBank** (Zhong et al., 2024): structured long-term memory for LLMs with forgetting curves inspired by Ebbinghaus — https://arxiv.org/abs/2305.10250

---

*Workshop complete. The agent now remembers.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions — not the only correct answers.

### Exercise 1 — Explore the Store

**What good output looks like:**

Semantic search should return the correct fact even when the query does not share exact keywords with the stored text. For example, querying `"what city does the user live in?"` should retrieve `"Bob lives in Tokyo"` even though the stored text says "Tokyo" not "city".

**If retrieval returns the wrong fact:** the query may be too vague, or the stored facts may be too similar to each other. Try making queries more specific, or adding `limit=1` to see only the top match.

In [ ]:
# Sample solution for Exercise 1
ans1_store = InMemoryStore()
bob_ans_ns = ("memories", "user-bob")

ans1_store.put(bob_ans_ns, "job", {"fact": "Bob is a data engineer at a fintech company"})
ans1_store.put(bob_ans_ns, "location", {"fact": "Bob lives in Tokyo, Japan"})
ans1_store.put(bob_ans_ns, "hobby", {"fact": "Bob brews craft beer on weekends"})
ans1_store.put(bob_ans_ns, "style", {"fact": "Bob prefers concise technical answers with code examples"})
ans1_store.put(bob_ans_ns, "quirk", {"fact": "Bob always asks for the worst-case complexity of any algorithm"})

for q in [
    "what does the user do for work?",
    "where does the user live?",
    "what are the user's hobbies?",
]:
    top = ans1_store.search(bob_ans_ns, query=q, limit=1)
    print(f"Q: {q}")
    print(f"A: {top[0].value['fact'] if top else '(no result)'}")
    print()

### Exercise 2 — Add a Third Thread

**Key insight to look for:** Thread-3 should retrieve memories stored in Threads 1 and 2 — facts it has no direct access to through graph state. If `result_t3["memories"]` is empty, check that you are using the same `graph_store` instance and the same `NAMESPACE` tuple. The store and namespace must be shared across all threads.

In [ ]:
# Sample solution for Exercise 2
# Uses the same `app` and `graph_store` from Part 4-5.

result_t3 = app.invoke(
    {
        "thread_id": "thread-3",
        "messages": [
            "I'm thinking of starting a blog about agentic AI. Any tips?",
        ],
        "memories": [],
        "response": "",
    }
)

print("Thread-3 memories retrieved:", result_t3["memories"])
print()
print("Thread-3 response:", result_t3["response"][:400])
print()

all_facts_after = graph_store.search(NAMESPACE, query="everything", limit=50)
print(f"Total facts after 3 threads: {len(all_facts_after)}")

### Exercise 3 — Build a Third User

**What the isolation check verifies:** `multi_store.search(("memories", "user-alice"), ...)` can only return facts stored under Alice's namespace. Carol's facts are stored under `("memories", "user-carol")` — a completely different partition of the store. The assert should always pass if the namespace pattern is applied consistently.

**If the assert fails:** check that `build_user_app` is passing the correct `user_id` to every `store.put` and `store.search` call. A common mistake is capturing `user_id` by reference rather than by value in a closure.

In [ ]:
# Sample solution for Exercise 3
carol_app = build_user_app("user-carol", multi_store)

# Introduction thread
carol_app.invoke(
    {
        "thread_id": "carol-intro",
        "messages": ["Hi, I'm Carol. I'm a data scientist who loves hiking in the Alps."],
        "memories": [],
        "response": "",
    }
)

# Follow-up thread
carol_result = carol_app.invoke(
    {
        "thread_id": "carol-follow",
        "messages": ["What's a good Python library for geospatial data analysis?"],
        "memories": [],
        "response": "",
    }
)
print("Carol's memories:", carol_result["memories"])
print("Carol's response:", carol_result["response"][:300])
print()

# Isolation check
alice_facts = multi_store.search(("memories", "user-alice"), query="hiking data science", limit=5)
assert all("Carol" not in f.value["fact"] for f in alice_facts), "Isolation breach!"
print("Isolation check passed: Carol's facts are not in Alice's namespace.")